# Vilier Component Pipeline

Run cells in order. Each component cell has an `*_input` prompt; press Enter to use the previous component output, or paste a path to resume from an artifact.

In [ ]:
import json
from pathlib import Path

import numpy as np

from pipeline.asr import export_speaker_asr_audio, load_asr_runner, transcribe_asr_segments, write_transcript_json
from pipeline.audio import duration_seconds, iter_audio_files, load_mono, write_wav
from pipeline.cli import _component_summary, write_resolved_config
from pipeline.diarization import DiariZenDiarizer, PyannotePixitDiarizer, build_diarization_chunks, build_speaker_linking_artifact, load_diarizer, write_speaker_linking_artifact
from pipeline.labeling import label_transcripts, load_labeling_runner, resolve_state_dir, write_state_outputs
from pipeline.music import apply_music_separation, load_music_separator
from pipeline.overlap_separation import apply_overlap_separation, load_overlap_separator
from pipeline.schema import SpeakerSegment, SpeakerTrack
from pipeline.timeline import annotate_overlaps, export_audacity_labels, export_segments_and_tracks, write_manifest
from pipeline.vad import SileroVadRunner, export_vad_audio, write_vad_txt

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def write_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding='utf-8')
    return path

def segment_from_dict(item):
    return SpeakerSegment(
        id=str(item['id']),
        speaker=str(item['speaker']),
        start=float(item['start']),
        end=float(item['end']),
        segment_wav=str(item.get('segment_wav', '')),
        is_overlap=bool(item.get('is_overlap', False)),
        overlap_group_id=item.get('overlap_group_id'),
    )

def segments_from_json(path):
    return [segment_from_dict(item) for item in read_json(path)]

def segments_to_json(segments):
    return [segment.to_manifest() for segment in segments]

def tracks_from_json(path):
    return [SpeakerTrack(id=str(item['id']), track_wav=str(item['track_wav'])) for item in read_json(path)]

def tracks_to_json(tracks):
    return [track.__dict__ for track in tracks]

def ask_path(prompt, default):
    value = input(f'{prompt} [{default}]: ').strip()
    return Path(value or default).expanduser()

def pass_msg(component, **items):
    print(f'{component} PASS')
    for key, value in items.items():
        print(f'  {key}={value}')

config_path = ask_path('config_input', 'config.json')
config = read_json(config_path)
dry_run = (input(f"dry_run [config={config.get('runtime', {}).get('dry_run', False)}]: ").strip().lower() or str(config.get('runtime', {}).get('dry_run', False)).lower()) in {'1', 'true', 'yes', 'y'}
raw_input_path = ask_path('raw_audio_input', config['entrypoint']['input_path'])
audio_files = iter_audio_files(raw_input_path)
if not audio_files:
    raise FileNotFoundError(f'No audio files found in {raw_input_path}')
for index, audio_file in enumerate(audio_files):
    print(f'{index}: {audio_file}')
audio_index = int(input('audio_index [0]: ').strip() or '0')
source_audio_path = audio_files[audio_index].resolve()
output_root = ask_path('output_root', config['entrypoint']['output_path']).resolve()
audio_id = source_audio_path.stem
output_dir = output_root / audio_id
output_dir.mkdir(parents=True, exist_ok=True)
sample_rate = int(config['entrypoint'].get('sample_rate', 16000))
state_dir = resolve_state_dir(config)
audio_state_dir = state_dir if state_dir.is_absolute() else output_dir / state_dir
standardized_audio_path = output_dir / 'audio.standardized.wav'
vad_json_path = output_dir / 'vad.json'
vad_audio_json_path = output_dir / 'vad_audio.json'
diarization_chunks_path = output_dir / 'diarization_chunks.json'
segments_path = output_dir / 'segments.json'
active_audio_path = standardized_audio_path
separation_audio_path = active_audio_path
speaker_tracks_path = output_dir / 'speaker_tracks.json'
asr_segments_path = output_dir / 'asr_segments.json'
transcript_path = output_dir / 'transcript.json'
vad_runner = None
diarizer = None
music_separator = None
overlap_separator = None
asr_runner = None
labeling_runner = None
segment_audio_overrides = {}
audacity_labels = {}
pass_msg('setup', audio=source_audio_path, output_dir=output_dir, sample_rate=sample_rate, dry_run=dry_run)


In [ ]:
preprocess_input = input(f'preprocess_input [{source_audio_path}]: ').strip()
preprocess_path = Path(preprocess_input or source_audio_path).expanduser().resolve()
waveform, sample_rate = load_mono(preprocess_path, sample_rate)
standardized_audio_path = output_dir / 'audio.standardized.wav'
write_wav(standardized_audio_path, waveform, sample_rate)
duration_sec = len(waveform) / sample_rate
pass_msg('preprocess', input=preprocess_path, output=standardized_audio_path, duration_sec=round(duration_sec, 3))


In [ ]:
vad_input = input(f'vad_input [{standardized_audio_path}]: ').strip()
vad_audio_input = Path(vad_input or standardized_audio_path).expanduser().resolve()
waveform, sample_rate = load_mono(vad_audio_input, sample_rate)
vad_runner = SileroVadRunner(config.get('vad', {}), sample_rate, dry_run=dry_run)
vad_segments = vad_runner.detect(waveform)
vad_json_path = write_json(output_dir / 'vad.json', vad_segments)
write_vad_txt(output_dir / 'vad.txt', vad_segments)
write_vad_txt(output_dir / 'labels' / 'vad.txt', vad_segments)
vad_audio_paths = export_vad_audio(waveform, sample_rate, vad_segments, output_dir)
vad_audio_json_path = write_json(output_dir / 'vad_audio.json', [str(path) for path in vad_audio_paths])
pass_msg('vad', input=vad_audio_input, segments=len(vad_segments), vad_json=vad_json_path, vad_audio=output_dir / 'vad_audio')


In [ ]:
diarization_chunks_input = input(f'diarization_chunks_input [{vad_json_path}]: ').strip()
chunks_vad_path = Path(diarization_chunks_input or vad_json_path).expanduser().resolve()
chunks_audio_input = input(f'diarization_chunks_audio_input [{standardized_audio_path}]: ').strip()
chunks_audio_path = Path(chunks_audio_input or standardized_audio_path).expanduser().resolve()
vad_segments = read_json(chunks_vad_path)
waveform, sample_rate = load_mono(chunks_audio_path, sample_rate)
diarization_chunks = build_diarization_chunks(
    waveform,
    sample_rate,
    vad_segments,
    output_dir,
    max_chunk_seconds=float(config.get('diarization', {}).get('max_chunk_seconds', 180.0)),
)
diarization_chunks_path = write_json(output_dir / 'diarization_chunks.json', diarization_chunks)
pass_msg('diarization_chunks', chunks=len(diarization_chunks), output=diarization_chunks_path)


In [ ]:
diarization_input = input(f'diarization_input [{standardized_audio_path}]: ').strip()
diarization_audio_path = Path(diarization_input or standardized_audio_path).expanduser().resolve()
diarization_vad_input = input(f'diarization_vad_input [{vad_json_path}]: ').strip()
diarization_vad_path = Path(diarization_vad_input or vad_json_path).expanduser().resolve()
diarization_chunks_override = input(f'diarization_chunks_json_input [{diarization_chunks_path}]: ').strip()
diarization_chunks_path = Path(diarization_chunks_override or diarization_chunks_path).expanduser().resolve()
vad_segments = read_json(diarization_vad_path)
diarization_chunks = read_json(diarization_chunks_path)
diarizer = load_diarizer(config.get('diarization', {}), dry_run=dry_run)
if isinstance(diarizer, (PyannotePixitDiarizer, DiariZenDiarizer)):
    segments = diarizer.diarize(diarization_audio_path, vad_segments)
else:
    segments = diarizer.diarize_chunks(diarization_chunks)
segments = annotate_overlaps(segments, float(config.get('overlap', {}).get('threshold_seconds', 0.05)))
segments_path = write_json(output_dir / 'segments.json', segments_to_json(segments))
speaker_linking_payload = build_speaker_linking_artifact(
    backend=str(config.get('diarization', {}).get('backend', '')),
    model=str(config.get('diarization', {}).get('model', '')),
    segments=segments,
    chunks=diarization_chunks,
    diarizer=diarizer,
)
speaker_linking_summary = write_speaker_linking_artifact(output_dir, speaker_linking_payload)
speaker_linking_summary_path = write_json(output_dir / 'speaker_linking.summary.json', speaker_linking_summary)
pass_msg('diarization', input=diarization_audio_path, segments=len(segments), output=segments_path, speaker_linking=speaker_linking_summary.get('audio'))


In [ ]:
music_input = input(f'music_input [{standardized_audio_path}]: ').strip()
music_audio_path = Path(music_input or standardized_audio_path).expanduser().resolve()
waveform, sample_rate = load_mono(music_audio_path, sample_rate)
music_separator = load_music_separator(config.get('music_separation', {}), dry_run=dry_run)
music_result = apply_music_separation(waveform, sample_rate, output_dir, music_separator)
music_waveform = music_result['waveform']
active_audio_path = output_dir / 'music_cleaned.wav' if music_result.get('applied') else standardized_audio_path
music_result_path = write_json(output_dir / 'music_separation.json', {key: value for key, value in music_result.items() if key != 'waveform'})
pass_msg('music_separation', input=music_audio_path, applied=music_result.get('applied'), active_audio=active_audio_path, output=music_result_path)


In [ ]:
separation_input = input(f'separation_input [{active_audio_path}]: ').strip()
separation_audio_path = Path(separation_input or active_audio_path).expanduser().resolve()
separation_segments_input = input(f'separation_segments_input [{segments_path}]: ').strip()
separation_segments_path = Path(separation_segments_input or segments_path).expanduser().resolve()
waveform, sample_rate = load_mono(separation_audio_path, sample_rate)
segments = segments_from_json(separation_segments_path)
overlap_separator = load_overlap_separator(config.get('overlap_separation', {}), dry_run=dry_run)
overlap_result = apply_overlap_separation(
    waveform,
    sample_rate,
    segments,
    overlap_separator,
    float(config.get('overlap_separation', {}).get('overlap_threshold_seconds', 0.2)),
    output_dir=output_dir,
)
segment_audio_overrides = overlap_result.get('segment_audio', {})
overlap_summary = {
    'enabled': bool(config.get('overlap_separation', {}).get('enabled', False)),
    'overlap_regions': overlap_result.get('overlap_regions', []),
    'enhanced_segment_count': len(segment_audio_overrides),
}
overlap_result_path = write_json(output_dir / 'overlap_separation.json', overlap_summary)
pass_msg('overlap_separation', input=separation_audio_path, regions=len(overlap_summary['overlap_regions']), enhanced_segments=len(segment_audio_overrides), output=overlap_result_path)


In [ ]:
tracks_input = input(f'tracks_input [{separation_audio_path}]: ').strip()
tracks_audio_path = Path(tracks_input or separation_audio_path).expanduser().resolve()
tracks_segments_input = input(f'tracks_segments_input [{segments_path}]: ').strip()
tracks_segments_path = Path(tracks_segments_input or segments_path).expanduser().resolve()
tracks_vad_input = input(f'tracks_vad_input [{vad_json_path}]: ').strip()
tracks_vad_path = Path(tracks_vad_input or vad_json_path).expanduser().resolve()
waveform, sample_rate = load_mono(tracks_audio_path, sample_rate)
segments = segments_from_json(tracks_segments_path)
vad_segments = read_json(tracks_vad_path)
speaker_tracks = export_segments_and_tracks(
    waveform,
    sample_rate,
    segments,
    output_dir,
    write_segment_wavs=bool(config.get('export', {}).get('write_segment_wavs', False)),
    segment_audio_overrides=globals().get('segment_audio_overrides', {}),
)
audacity_labels = export_audacity_labels(output_dir, segments, vad_segments)
track_vad_runner = SileroVadRunner(config.get('vad', {}), sample_rate, dry_run=dry_run)
asr_segments = export_speaker_asr_audio(output_dir, speaker_tracks, track_vad_runner)
speaker_tracks_path = write_json(output_dir / 'speaker_tracks.json', tracks_to_json(speaker_tracks))
audacity_labels_path = write_json(output_dir / 'audacity_labels.json', {key: str(value) for key, value in audacity_labels.items()})
asr_segments_path = write_json(output_dir / 'asr_segments.json', asr_segments)
pass_msg('tracks', speakers=len(speaker_tracks), asr_segments=len(asr_segments), tracks=speaker_tracks_path, asr_segments_json=asr_segments_path)


In [ ]:
asr_input = input(f'asr_input [{asr_segments_path}]: ').strip()
asr_segments_path = Path(asr_input or asr_segments_path).expanduser().resolve()
asr_segments = read_json(asr_segments_path)
asr_runner = load_asr_runner(config, dry_run=dry_run)
if asr_runner is None:
    transcript = []
else:
    transcript = transcribe_asr_segments(output_dir, asr_segments, asr_runner)
transcript_path = output_dir / 'transcript.json'
write_transcript_json(transcript_path, transcript)
pass_msg('asr', enabled=asr_runner is not None, transcripts=len(transcript), output=transcript_path)


In [ ]:
state_labeling_input = input(f'state_labeling_input [{transcript_path}]: ').strip()
state_transcript_path = Path(state_labeling_input or transcript_path).expanduser().resolve()
transcript = read_json(state_transcript_path) if state_transcript_path.exists() else []
labeling_runner = load_labeling_runner(config, dry_run=dry_run)
if labeling_runner is None:
    labeled_transcript = transcript
    state_summary = {'enabled': False, 'labeled': 0}
else:
    labeled_transcript = label_transcripts(transcript, labeling_runner)
    write_transcript_json(transcript_path, labeled_transcript)
    state_summary = write_state_outputs(audio_id, output_dir, audio_state_dir, labeled_transcript)
state_labeling_path = write_json(output_dir / 'state_labeling.json', state_summary)
pass_msg('state_labeling', enabled=labeling_runner is not None, labeled=state_summary.get('labeled', 0), output=state_labeling_path)


In [ ]:
manifest_input = input(f'manifest_input [{output_dir}]: ').strip()
manifest_output_dir = Path(manifest_input or output_dir).expanduser().resolve()
segments = segments_from_json(manifest_output_dir / 'segments.json')
vad_segments = read_json(manifest_output_dir / 'vad.json')
speaker_tracks = tracks_from_json(manifest_output_dir / 'speaker_tracks.json')
asr_segments = read_json(manifest_output_dir / 'asr_segments.json') if (manifest_output_dir / 'asr_segments.json').exists() else []
transcript = read_json(manifest_output_dir / 'transcript.json') if (manifest_output_dir / 'transcript.json').exists() else []
diarization_chunks = read_json(manifest_output_dir / 'diarization_chunks.json') if (manifest_output_dir / 'diarization_chunks.json').exists() else []
vad_audio_paths = [Path(path) for path in read_json(manifest_output_dir / 'vad_audio.json')] if (manifest_output_dir / 'vad_audio.json').exists() else []
speaker_linking_summary = read_json(manifest_output_dir / 'speaker_linking.summary.json') if (manifest_output_dir / 'speaker_linking.summary.json').exists() else {'audio': 'speaker_linking.json'}
music_summary = read_json(manifest_output_dir / 'music_separation.json') if (manifest_output_dir / 'music_separation.json').exists() else {'enabled': False, 'applied': False}
overlap_summary = read_json(manifest_output_dir / 'overlap_separation.json') if (manifest_output_dir / 'overlap_separation.json').exists() else {'enabled': False, 'overlap_regions': []}
state_summary = read_json(manifest_output_dir / 'state_labeling.json') if (manifest_output_dir / 'state_labeling.json').exists() else {'enabled': False}
components = {
    'vad': _component_summary(config.get('vad', {}), True, vad_runner),
    'diarization': _component_summary(config.get('diarization', {}), True, diarizer),
    'music_separation': _component_summary(config.get('music_separation', {}), bool(config.get('music_separation', {}).get('enabled', False)), music_separator),
    'overlap_separation': _component_summary(config.get('overlap_separation', {}), bool(config.get('overlap_separation', {}).get('enabled', False)), overlap_separator, model_key='model_name'),
    'asr': _component_summary(config.get('asr', {}), bool(config.get('asr', {}).get('enabled', False)), asr_runner),
    'state_labeling': _component_summary(config.get('state_labeling', {}), bool(config.get('state_labeling', {}).get('enabled', False)), labeling_runner),
}
run_config = write_resolved_config(manifest_output_dir, config, input_path=source_audio_path, output_root=output_root, dry_run=dry_run, sample_rate=sample_rate, components=components)
manifest_path = write_manifest(
    manifest_output_dir,
    audio_id,
    source_audio_path,
    manifest_output_dir / 'audio.standardized.wav',
    duration_seconds(manifest_output_dir / 'audio.standardized.wav'),
    sample_rate,
    segments,
    speaker_tracks,
    vad_segments,
    audacity_labels=globals().get('audacity_labels', {}),
    vad_audio=vad_audio_paths,
    asr_segments=asr_segments,
    diarization_chunks=diarization_chunks,
    speaker_linking=speaker_linking_summary,
    run_config=run_config,
    music_separation=music_summary,
    overlap_separation=overlap_summary,
    transcript=transcript,
    state_labeling=state_summary,
)
pass_msg('manifest', output=manifest_path)
